# 🚀 SignFlow: Full 2,000-Class WLASL Model Training on Google Colab GPU

This notebook trains the **Robust Holistic Fusion Model** across the **entire 2,000-class WLASL vocabulary** (21,083 samples) using Google Colab's free T4 GPU.

### Pipeline Overview:
1. **GPU Check** (Tesla T4 15 GB VRAM verification)
2. **Mount Google Drive** & unpack all 21,083 holistic landmarks (`landmarks_all.tar.gz`)
3. **Install Dependencies** (`torch`, `pyyaml`, `scikit-learn`, `matplotlib`)
4. **Train 2,000-Class Model** (`configs/experiments/wlasl2000_proposed.yaml`)
5. **Plot Training & Validation Metrics** (Cross-Entropy Loss, Macro-F1, Accuracy)
6. **Held-Out Test Set Evaluation** (Top-1 & Top-5 accuracy across all 2,000 classes)
7. **Robustness Benchmark** (Spatial coordinate jitter & temporal frame loss resilience)
8. **Export Checkpoints & Results** to Google Drive

--- 
## Step 1: Verify Colab GPU Acceleration
Ensure your Colab runtime is set to GPU: **Runtime > Change runtime type > T4 GPU**.

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")
else:
    print("⚠️ WARNING: GPU not detected. Please select Runtime > Change runtime type > T4 GPU.")

--- 
## Step 2: Mount Google Drive & Unpack Data onto Fast Local NVMe SSD

Make sure your Google Drive folder `ASL_Colab` contains:
* `asl_code.tar.gz` (3.2 MB - source code, manifests, and configs)
* `landmarks_all.tar.gz` (all 21,083 landmark files for 2,000-class training)
*(Alternatively, if you are testing on 100 or 20 classes, you can use `landmarks_wlasl100.tar.gz` or `landmarks_wlasl20.tar.gz`)*

In [ ]:
import os
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define source and destination directories
DRIVE_DIR = '/content/drive/MyDrive/ASL_Colab'
LOCAL_PROJECT = '/content/asl_project'

os.makedirs(LOCAL_PROJECT, exist_ok=True)
%cd {LOCAL_PROJECT}

# 3. Unpack Code
code_archive = os.path.join(DRIVE_DIR, 'asl_code.tar.gz')
if os.path.exists(code_archive):
    print("Unpacking code archive...")
    !tar -xzf "{code_archive}" -C {LOCAL_PROJECT}
    print("✓ Code unpacked successfully.")
else:
    print(f"⚠️ {code_archive} not found! Check your Drive path.")

# 4. Unpack Landmark Dataset
lm_all = os.path.join(DRIVE_DIR, 'landmarks_all.tar.gz')
lm_100 = os.path.join(DRIVE_DIR, 'landmarks_wlasl100.tar.gz')
lm_20  = os.path.join(DRIVE_DIR, 'landmarks_wlasl20.tar.gz')

if os.path.exists(lm_all):
    print("Unpacking ALL 21,083 landmarks (~1-2 minutes)... Please wait.")
    !tar -xzf "{lm_all}" -C {LOCAL_PROJECT}
    print("✓ All 2,000-class landmarks unpacked successfully!")
elif os.path.exists(lm_100):
    print("Unpacking WLASL-100 landmarks...")
    !tar -xzf "{lm_100}" -C {LOCAL_PROJECT}
    print("✓ WLASL-100 landmarks unpacked successfully.")
elif os.path.exists(lm_20):
    print("Unpacking WLASL-20 landmarks...")
    !tar -xzf "{lm_20}" -C {LOCAL_PROJECT}
    print("✓ WLASL-20 landmarks unpacked successfully.")
else:
    print("⚠️ No landmark archive found in Drive folder 'ASL_Colab'.")

# 5. Check local file count
lm_count = len(os.listdir('data/landmarks')) if os.path.exists('data/landmarks') else 0
print(f"\nTotal local landmark files ready: {lm_count} / 21,083")

--- 
## Step 3: Install Dependencies & Setup Environment

In [ ]:
# Install project requirements and install package in editable mode
!pip install -q pyyaml pydantic scikit-learn matplotlib seaborn
!pip install -q -e .

# Set PYTHONPATH environment variable for all notebook subshells
import sys
sys.path.insert(0, '/content/asl_project/src')
%env PYTHONPATH=/content/asl_project/src
print('✓ Environment dependencies and sign_language package ready.')

--- 
## Step 4: Train the 2,000-Class Model on GPU

### Recommended Training Setup for 2,000 Classes:
- **Architecture**: `RobustHolisticFusionClassifier` (256 hidden units, 128 embedding dim, 2 layers, 0.3 dropout)
- **Batch Size**: 32 (balances gradient estimation and GPU VRAM on T4)
- **Epochs**: 80–100
- **Dataset**: 14,289 training samples, 3,916 validation samples across all 2,000 classes

In [ ]:
# @title Accelerated Training Configuration Form
config_file = "configs/colab/wlasl100_max_accuracy.yaml" # @param ["configs/colab/wlasl100_max_accuracy.yaml", "configs/colab/wlasl100_reliability_fusion_holistic.yaml", "configs/colab/wlasl100_tcn_holistic.yaml", "configs/experiments/wlasl100_proposed.yaml", "configs/colab/wlasl100_baseline_colab.yaml", "configs/colab/wlasl100_reliability_fusion_colab.yaml", "configs/experiments/wlasl2000_proposed.yaml"]
protocol = "official" # @param ["official", "signer_disjoint"]
epochs = 100 # @param {type:"integer"}
batch_size = 64 # @param [16, 32, 64, 128]
num_workers = 2 # @param {type:"integer"}
val_interval = 2 # @param {type:"integer"}
resume_checkpoint = "" # @param {type:"string"}
device = "cuda" # @param ["cuda", "cpu"]

import os
if protocol == "signer_disjoint":
    train_manifest = "data/manifests/signer_disjoint/train_manifest.json"
    val_manifest = "data/manifests/signer_disjoint/validation_manifest.json"
else:
    train_manifest = "data/manifests/train_manifest.json"
    val_manifest = "data/manifests/validation_manifest.json"

resume_flag = f"--resume {resume_checkpoint}" if (resume_checkpoint and os.path.exists(resume_checkpoint)) else ""
drive_backup = "/content/drive/MyDrive/ASL_Colab/checkpoints"

print(f"Starting accelerated training ({protocol} protocol) for {epochs} epochs on {device} (batch={batch_size}, workers={num_workers})...")
!PYTHONPATH=/content/asl_project/src python scripts/train.py \
    --config {config_file} \
    --epochs {epochs} \
    --batch-size {batch_size} \
    --num-workers {num_workers} \
    --val-interval {val_interval} \
    --drive-backup {drive_backup} \
    --device {device} \
    --manifest {train_manifest} \
    --validation-manifest {val_manifest} \
    --landmarks-dir data/landmarks \
    --checkpoint-dir models {resume_flag}


--- 
## Step 5: Plot Training Loss & Validation Macro-F1 Progression

In [ ]:
import json
import os
import matplotlib.pyplot as plt

metrics_file = 'results/training_metrics.json'
if os.path.exists(metrics_file):
    with open(metrics_file, 'r') as f:
        history = json.load(f)

    train_losses = history.get('loss_history') or history.get('train_loss', [])
    epochs_range = range(1, len(train_losses) + 1)

    val_history = history.get('validation_history', [])
    if val_history and isinstance(val_history[0], dict):
        val_macro_f1 = [v.get('macro_f1', 0.0) for v in val_history]
        val_acc = [v.get('accuracy', 0.0) for v in val_history]
    else:
        val_macro_f1 = history.get('val_macro_f1', [])
        val_acc = history.get('val_accuracy', [])

    plt.figure(figsize=(14, 5))

    # Loss curve
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, train_losses, label='Train Loss', color='#2563eb', lw=2)
    plt.xlabel('Epoch')
    plt.ylabel('Cross-Entropy Loss')
    plt.title('Training Loss Progression')
    plt.grid(True, alpha=0.3)
    plt.legend()

    # Macro-F1 & Accuracy curve
    plt.subplot(1, 2, 2)
    if val_macro_f1:
        val_x = range(1, len(val_macro_f1) + 1)
        plt.plot(val_x, val_macro_f1, label='Validation Macro-F1', color='#10b981', lw=2)
    if val_acc:
        val_x = range(1, len(val_acc) + 1)
        plt.plot(val_x, val_acc, label='Validation Accuracy', color='#f59e0b', lw=1.5, ls='--')
    plt.xlabel('Validation Evaluation Step')
    plt.ylabel('Score')
    plt.title('Validation Performance')
    plt.grid(True, alpha=0.3)
    plt.legend()

    plt.tight_layout()
    os.makedirs('results', exist_ok=True)
    plt.savefig('results/training_curves.png', dpi=200)
    plt.show()
    print('✓ Plot saved to results/training_curves.png')
else:
    print(f'⚠️ {metrics_file} not found yet.')


---
## Step 6: Held-Out Test Evaluation

- **Protocol A (Official WLASL)**: Uses `data/manifests/test_manifest.json` (benchmark comparison against published literature; note: signers overlap with train).
- **Protocol B (Signer-Disjoint)**: Uses `data/manifests/signer_disjoint/test_manifest.json` (strict out-of-distribution evaluation; zero signer overlap).


In [ ]:
test_manifest = 'data/manifests/test_manifest.json' if protocol == 'official' else 'data/manifests/signer_disjoint/test_manifest.json'
report_output = 'results/evaluation_report.json'

!PYTHONPATH=/content/asl_project/src python scripts/evaluate.py \
    --config {config_file} \
    --checkpoint models/robust_holistic_fusion_best.pt \
    --manifest {test_manifest} \
    --landmarks-dir data/landmarks \
    --output {report_output} \
    --protocol {protocol} \
    --device {device}

if os.path.exists(report_output):
    with open(report_output) as f:
        eval_report = json.load(f)
    
    metrics = eval_report.get('metrics', eval_report)
    top1 = metrics.get('top1_accuracy', metrics.get('accuracy', 0.0))
    top5 = metrics.get('top5_accuracy', 0.0)
    top10 = metrics.get('top10_accuracy', 0.0)
    macro_f1 = metrics.get('macro_f1', 0.0)
    weighted_f1 = metrics.get('weighted_f1', 0.0)
    balanced_acc = metrics.get('balanced_accuracy', 0.0)
    ece = metrics.get('expected_calibration_error', 0.0)
    total_samples = eval_report.get('diagnostics', {}).get('evaluated_samples', eval_report.get('total_samples', 0))
    
    print('\n' + '='*60)
    print(f'EVALUATION REPORT — Protocol: {protocol.upper()}')
    print('='*60)
    print(f"Top-1 Accuracy:     {top1:.4f} ({top1*100:.2f}%)")
    print(f"Top-5 Accuracy:     {top5:.4f} ({top5*100:.2f}%)")
    print(f"Top-10 Accuracy:    {top10:.4f} ({top10*100:.2f}%)")
    print(f"Balanced Accuracy:  {balanced_acc:.4f}")
    print(f"Macro-F1:           {macro_f1:.4f}")
    print(f"Weighted-F1:        {weighted_f1:.4f}")
    print(f"Expected Calib Err: {ece:.4f}")
    print(f"Evaluated Samples:  {total_samples}")
    print('='*60)


--- 
## Step 7: Robustness Stress-Testing (Noise & Disturbances)

In [ ]:
!PYTHONPATH=/content/asl_project/src python scripts/evaluate_robustness.py \
    --config {config_file} \
    --checkpoint models/robust_holistic_fusion_best.pt \
    --manifest data/manifests/test_manifest.json \
    --landmarks-dir data/landmarks \
    --output results/robustness/wlasl2000_robustness_report.json \
    --device cuda

rob_file = 'results/robustness/wlasl2000_robustness_report.json'
if os.path.exists(rob_file):
    with open(rob_file) as f:
        rob = json.load(f)
    print(f"\nClean Accuracy: {rob.get('clean_accuracy', 0):.4f}")
    print('\nDisturbance Stress Tests:')
    print(f"{'Disturbance':<30} | {'Severity':<8} | {'Noisy Acc':<10} | {'Degradation':<12}")
    print('-' * 70)
    for entry in rob.get('scenarios', [])[:12]:
        print(f"{entry['scenario']:<30} | {entry['severity']:<8.1f} | {entry['accuracy']:<10.4f} | {entry['degradation']:<12.4f}")


--- 
## Step 8: Save 2,000-Class Model Checkpoint & Metrics to Google Drive

In [ ]:
import shutil

TARGET_BACKUP = '/content/drive/MyDrive/ASL_Colab/results_2000'
os.makedirs(TARGET_BACKUP, exist_ok=True)

# Copy model checkpoint
if os.path.exists('models/robust_holistic_fusion_best.pt'):
    shutil.copy2('models/robust_holistic_fusion_best.pt', os.path.join(TARGET_BACKUP, 'wlasl2000_robust_holistic_fusion_best.pt'))
    print("✓ 2,000-class model checkpoint saved to Google Drive!")

# Copy all results
if os.path.exists('results'):
    !cp -r results/* "{TARGET_BACKUP}/"
    print("✓ All evaluation reports, figures, and metrics saved to Google Drive!")

print(f"\n🎉 All artifacts are safely preserved in: {TARGET_BACKUP}")